# V021: separate R/C one-hot encoding and restricted mixup

Two distinct mechanisms inspired by first/third-place reports in the user-supplied synthesis. All labels used by mixup come from the current training minibatch. No dataset read, training, GPU allocation or submission occurs on definition. Requires reviewed V005/V008 components loaded first.


In [ ]:
import copy,json,hashlib
from pathlib import Path
import numpy as np
import torch
from torch import nn
V021_COMPONENT='encoding_mixup_components_v021.ipynb'
V021_MIX_VERSION='v021-RC-phase-matched-mixup-1'

class V021SeparateOneHotRNN(nn.Module):
    """Replace the two direct numerical R/C inputs by six fixed one-hot inputs.

    Receives legacy transformed numeric inputs followed by the internal joint
    code. The input preprocessor remains training-only. No labels enter encoding.
    """
    def __init__(self,spec,n_features):
        super().__init__();self.n_features=int(n_features)
        if self.n_features<2:raise ValueError('Expected direct R,C first')
        self.backbone=V5PressureRNN(spec,self.n_features+4)
    def encode(self,x):
        if x.ndim!=3 or x.shape[-1]!=self.n_features+1:raise ValueError('Unexpected raw transformed width')
        code=x[...,-1].long()
        if not bool(((code>=0)&(code<9)).all()):raise ValueError('Invalid joint R/C code')
        r=torch.nn.functional.one_hot(code//3,3).to(x.dtype);c=torch.nn.functional.one_hot(code%3,3).to(x.dtype)
        return torch.cat([r,c,x[...,2:self.n_features],x[...,-1:]],dim=-1)
    def forward(self,x):return self.backbone(self.encode(x))

def v21_onehot_factory(spec,n_features,prep):return V021SeparateOneHotRNN(spec,n_features)
def v21_plain_factory(spec,n_features,prep):return V5PressureRNN(spec,n_features)
for _f in [v21_onehot_factory,v21_plain_factory]:_f.__module__='encoding_mixup_components_v021'

def matched_mixup_v021(x,y,mask,feature_names,alpha=.2,probability=.5):
    """Training-batch-only, whole-breath interpolation; no validation is accepted.

    Partners share exact R/C and the full phase mask. A cyclic permutation of
    each shuffled compatible group guarantees no self-pair for groups of2+.
    CPU Torch randomness is part of the existing serialized RNG state.
    """
    if not np.isfinite(alpha) or alpha<=0 or not np.isfinite(probability) or not 0<=probability<=1:raise ValueError('Invalid mixup settings')
    if x.ndim!=3 or y.shape!=x.shape[:2] or mask.shape!=y.shape or len(feature_names)!=x.shape[-1]-1:raise ValueError('Mixup shapes or feature names differ')
    if list(feature_names[:2])!=['R','C']:raise ValueError('Direct numerical R,C must be first')
    n=len(x);empty=dict(eligible_breaths=0,mixed_breaths=0,total_breaths=n)
    if probability==0:return x,y,empty
    joint=x[:,0,-1].detach().cpu().long().numpy();phases=mask.detach().cpu().numpy().astype(bool);groups={}
    if not bool((x[...,-1]==x[:,0,-1,None]).all()):raise ValueError('R/C changes within a breath')
    for i in range(n):groups.setdefault((int(joint[i]),phases[i].tobytes()),[]).append(i)
    partner=torch.arange(n);eligible=torch.zeros(n,dtype=torch.bool)
    for indices in groups.values():
        if len(indices)<2:continue
        indices=torch.tensor(indices,dtype=torch.long);shuffled=indices[torch.randperm(len(indices))]
        partner[shuffled]=shuffled.roll(1);eligible[indices]=True
    active=eligible&(torch.rand(n)<probability);lam=torch.distributions.Beta(float(alpha),float(alpha)).sample((n,));lam=torch.maximum(lam,1-lam);lam[~active]=1.
    effective=active&(lam<1.)
    stats=dict(eligible_breaths=int(eligible.sum()),mixed_breaths=int(effective.sum()),total_breaths=n)
    if not bool(effective.any()):return x,y,stats
    index=partner.to(x.device);weight=lam.to(device=x.device,dtype=x.dtype)
    mixed_x=weight[:,None,None]*x+(1-weight[:,None,None])*x[index]
    mixed_y=weight[:,None]*y+(1-weight[:,None])*y[index]
    protected=[feature_names.index(k) for k in ['R','C','u_out','step'] if k in feature_names]
    mixed_x[...,protected]=x[...,protected];mixed_x[...,-1]=x[...,-1]
    return mixed_x,mixed_y,stats

def _v21_mix_metadata(alpha,probability,sources):
    if not np.isfinite(alpha) or alpha<=0 or not np.isfinite(probability) or not 0<=probability<=1:raise ValueError('Invalid mixup recipe')
    source=notebook_code_sha256_v008(_v8_notebook_path(V021_COMPONENT))
    if not sources or sources.get(V021_COMPONENT)!=source:raise ValueError('Exact encoding/mixup component pin required')
    return dict(version=V021_MIX_VERSION,alpha=float(alpha),probability=float(probability),component_code_sha256=source,source_fingerprints=dict(sorted(sources.items())),pairing='same R/C and entire phase mask; shuffled non-self cycle within training batch',one_coefficient_per_breath=True,protected_features=['R','C','u_out','step','internal_joint_code'],preprocessing='fit only original unaugmented training breaths',validation='unaugmented native inspiration-only point-weighted pressure MAE',loss_kind='curve_l1',online_training_metric_uses_augmented_targets=probability>0)

def _v21_mix_engine(arm,alpha,probability,sources,device):
    meta=_v21_mix_metadata(alpha,probability,sources);engine,base=_v8_engine(arm,'curve_l1',v21_plain_factory,sources,device)
    old_fp=engine['_fingerprint'];old_payload=engine['_checkpoint_payload'];old_prepare=engine['_prepare_run'];old_configuration=engine['_train_configuration'];latest={}
    def configuration(cfg):
        c=old_configuration(cfg)
        if c['residual_baseline']:raise ValueError('Mixup experiment requires one common linear target scale without varying baseline')
        return c
    def fingerprint(*args,**kwargs):
        r=old_fp(*args,**kwargs);r['v021_mixup']=copy.deepcopy(meta);return r
    def payload(model,prep,spec,cfg,seed,run_fingerprint,optimizer,scheduler,generator,epoch,history,completed=False,termination=None):
        if history and history[-1]['epoch']==epoch:history[-1].update(copy.deepcopy(latest))
        r=old_payload(model,prep,spec,cfg,seed,run_fingerprint,optimizer,scheduler,generator,epoch,history,completed,termination);r['v021_mixup']=copy.deepcopy(meta);return r
    def prepare(raw,y,out,spec,cfg,seed,feature_names,checkpoint_path,resume,run_fingerprint,selected_device):
        if resume:
            saved=torch.load(engine['_last_path'](checkpoint_path),map_location='cpu',weights_only=False)
            if saved.get('v021_mixup')!=meta or saved.get('fingerprint',{}).get('v021_mixup')!=meta:raise ValueError('Mixup resume source/recipe changed')
        return old_prepare(raw,y,out,spec,cfg,seed,feature_names,checkpoint_path,resume,run_fingerprint,selected_device)
    def training_epoch(model,loader,optimizer,cfg,prep,selected_device):
        model.train();total_error=0.;count=0;steps=0;adj_error=0.;adj_count=0;objective_sum=0.;eligible=0;mixed=0;breaths=0
        for xb,yb,mask in loader:
            xb,yb,mask=(v.to(selected_device,non_blocking=selected_device.type=='cuda') for v in (xb,yb,mask))
            if not bool(mask.any()):continue
            xb,yb,stats=matched_mixup_v021(xb,yb,mask,prep['feature_names'],alpha,probability)
            optimizer.zero_grad(set_to_none=True);prediction=model(xb);loss,main,adj=curve_l1_loss_v008(prediction,yb,mask)
            if not torch.isfinite(loss):raise RuntimeError('Nonfinite mixed training loss')
            loss.backward();nn.utils.clip_grad_norm_(model.parameters(),float(cfg['gradient_clip']));optimizer.step();steps+=1
            points=int(mask.sum());pairs=int((mask[:,1:]&mask[:,:-1]).sum());total_error+=float((prediction-yb).masked_select(mask).detach().abs().sum())*prep['target_scale'];count+=points
            adj_error+=float(adj.detach())*pairs*prep['target_scale'];adj_count+=pairs;objective_sum+=float(loss.detach())*points;eligible+=stats['eligible_breaths'];mixed+=stats['mixed_breaths'];breaths+=stats['total_breaths']
        if not count:raise ValueError('No inspired training points')
        latest.clear();latest.update(training_loss_kind='curve_l1',training_online_objective_scaled=objective_sum/count,training_adjacent_inspiratory_pairs=adj_count,training_online_increment_mae=adj_error/max(adj_count,1),training_mixup_eligible_breaths=eligible,training_mixup_mixed_breaths=mixed,training_mixup_total_breaths=breaths,training_online_metric_uses_augmented_targets=probability>0)
        return float(total_error/count),count,steps
    engine.update(_train_configuration=configuration,_fingerprint=fingerprint,_checkpoint_payload=payload,_prepare_run=prepare,_training_epoch=training_epoch)
    return engine,meta,base

def fit_validate_v021_mixup(x_train,y_train,u_out_train,x_val,y_val,u_out_val,model_spec,train_cfg,seed,checkpoint_path,epoch_log_path=None,feature_names=None,train_diagnostic_breaths=2048,device='cuda',resume=False,on_epoch=None,*,alpha=.2,probability=.5,arm='v021_mixup',source_fingerprints=None):
    e,m,b=_v21_mix_engine(arm,alpha,probability,source_fingerprints,device);r=e['fit_validate'](x_train,y_train,u_out_train,x_val,y_val,u_out_val,model_spec,train_cfg,seed,checkpoint_path,epoch_log_path=epoch_log_path,feature_names=feature_names,train_diagnostic_breaths=train_diagnostic_breaths,device=device,resume=resume,on_epoch=on_epoch);r=_v8_report(r,b);r['v021_mixup']=m;return r

def fit_fixed_v021_mixup(x_train,y_train,u_out_train,model_spec,train_cfg,seed,lr_schedule,checkpoint_path,epoch_log_path=None,feature_names=None,device='cuda',resume=False,on_epoch=None,*,alpha=.2,probability=.5,arm='v021_mixup',source_fingerprints=None):
    e,m,b=_v21_mix_engine(arm,alpha,probability,source_fingerprints,device);r=e['fit_fixed'](x_train,y_train,u_out_train,model_spec,train_cfg,seed,lr_schedule,checkpoint_path,epoch_log_path=epoch_log_path,feature_names=feature_names,device=device,resume=resume,on_epoch=on_epoch);r=_v8_report(r,b);r['v021_mixup']=m;return r

def load_checkpoint_v021_mixup(path,device='cpu',*,alpha=.2,probability=.5,expected_arm='v021_mixup',source_fingerprints=None):
    saved=torch.load(path,map_location='cpu',weights_only=False);m=_v21_mix_metadata(alpha,probability,source_fingerprints)
    if saved.get('v021_mixup')!=m or saved.get('fingerprint',{}).get('v021_mixup')!=m:raise ValueError('Mixup checkpoint recipe/source changed')
    return load_checkpoint_v008(path,device=device,model_factory=v21_plain_factory,expected_arm=expected_arm,expected_loss_kind='curve_l1',source_fingerprints=source_fingerprints)

def predict_checkpoint_v021_mixup(path,x,device='cpu',batch_size=512,*,alpha=.2,probability=.5,expected_arm='v021_mixup',source_fingerprints=None):
    if int(batch_size)<1:raise ValueError('Positive prediction batch size required')
    model,prep,saved=load_checkpoint_v021_mixup(path,device,alpha=alpha,probability=probability,expected_arm=expected_arm,source_fingerprints=source_fingerprints);e,_,_=_v21_mix_engine(expected_arm,alpha,probability,source_fingerprints,device)
    return e['_predict_native'](model,e['transform_inputs'](x,prep),prep,e['_sequence_device'](device),int(batch_size))
